# 00 · Pipeline anatomy — the static map

The mental model before any run: the **13 nodes** of the LangGraph pipeline,
the **conditional routers** that steer between them, the **15-agent roster**,
and the **7 document classes** with their specialists and typed extraction
schemas.

Everything below is *rendered from the live code* — the compiled graph object,
`graph.routing`'s router signatures, and `src/config/taxonomy.yaml` via
`pipeline.config`. Nothing is copy-pasted, so this map cannot drift from the
real wiring.

**Honesty label:** no LLM calls, no pipeline runs — pure introspection of the
real modules. Thresholds and rosters are exactly what the running system uses.

Companion module: `notebooks/pipeline_lab.py`. Plan of record: `notebooks/PLAN.md`.

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab

lab.quiet_logs()  # reproducible outputs: WARNING+ only

## The graph — 13 nodes

Each pipeline node wraps one or more agents. The **span name** column is the
verb-first name the node publishes to Langfuse (the contract The-Mailroom
visualizer renders).

In [2]:
from pipeline_lab import NODE_AGENT_ROLES

amap = lab.graph_map()
print(f"{len(amap['nodes'])} nodes:")
print()
for node in amap["nodes"]:
    span = amap["span_names"][node]
    agent, role = NODE_AGENT_ROLES.get(span, ("(routing/support)", "see routers below"))
    print(f"{node:16s} span={span:20s} agents={agent}")
    print(f"{'':16s} role: {role}")

13 nodes:

intake           span=intake-document      agents=pdf_transcriber / image_extractor (conditional)
                 role: Reads the raw file, extracts text + page images, assigns doc_id
classify         span=classify-document    agents=sorter (+ sorter_reviewer Lane A, judge-classification)
                 role: Labels doc_type + confidence; bands decide what happens next
retry_classify   span=classify-document    agents=sorter (+ sorter_reviewer Lane A, judge-classification)
                 role: Labels doc_type + confidence; bands decide what happens next
review_classify  span=classify-document    agents=sorter (+ sorter_reviewer Lane A, judge-classification)
                 role: Labels doc_type + confidence; bands decide what happens next
extract          span=extract-fields       agents=class specialist (+ judge/arbiter Lane B)
                 role: Fills the class schema's typed fields
retry_extract    span=extract-fields       agents=class specialist (+ judge/arbit

## The routers — where the graph decides

Conditional edges: each router is a pure function of the state returning the
next node. The choice lists below are extracted from the routers' own return
type hints.

In [3]:
for router, choices in amap["routers"].items():
    print(f"{router:26s} -> {', '.join(choices)}")

after_classify             -> classify, retry_classify, extract, human_review
after_retry_classify       -> retry_classify, review_classify, extract, human_review
after_extraction           -> extract, retry_extract, compile_report, human_review, boss_escalation
after_retry_extraction     -> retry_extract, compile_report, human_review, boss_escalation
after_extraction_gated     -> extract, retry_extract, compile_report, judge_verify, human_review, boss_escalation
after_retry_extraction_gated -> retry_extract, compile_report, judge_verify, human_review, boss_escalation
after_boss                 -> boss_escalation, compile_report, human_review
after_human_review         -> compile_report, failed
after_review_classify      -> review_classify, extract, human_review
after_judge                -> judge_verify, compile_report, arbiter, human_review
after_arbiter              -> arbiter, compile_report, retry_extract, human_review


## The confidence bands

The numbers that drive the routers, read live from the same config the
routers read (`taxonomy.yaml` → `pipeline.config`).

In [4]:
bands = lab.band_report()
print(f"high:            {bands['high']}   — at/above: confident, straight through")
print(f"low:             {bands['low']}   — below: not confident at all")
print(f"judge_band_high: {bands['judge_band_high']}   — extraction in [low, judge_band_high) triggers the judge (KANBAN-063)")
print()
print("classification bands:  conf >= high -> extract | low <= conf < high -> retry, then Lane A | conf < low -> human review")
print(f"extraction gate:       judge fires when low ({bands['low']}) <= extraction_conf < judge_band_high ({bands['judge_band_high']})")

high:            0.95   — at/above: confident, straight through
low:             0.7   — below: not confident at all
judge_band_high: 0.85   — extraction in [low, judge_band_high) triggers the judge (KANBAN-063)

classification bands:  conf >= high -> extract | low <= conf < high -> retry, then Lane A | conf < low -> human review
extraction gate:       judge fires when low (0.7) <= extraction_conf < judge_band_high (0.85)


## The 7 document classes

Each class: its specialist agent, its typed extraction schema, and its field
types (`entity_list:name`, `money`, `date`, `id`, …).

In [5]:
for cls in lab.taxonomy_table():
    print(f"{cls['key']:18s} {cls['label']}")
    print(f"{'':18s} specialist: {cls['specialist']}  schema: {cls['schema']}  fields: {len(cls['fields'])}")
    for fname, ftype in list(cls["fields"].items())[:4]:
        print(f"{'':20s} {fname}: {ftype}")
    more = len(cls["fields"]) - 4
    if more > 0:
        print(f"{'':20s} … +{more} more")
    print()

contract           Contract / Agreement
                   specialist: contracts_specialist  schema: ContractExtraction  fields: 8
                     parties: entity_list:name
                     effective_date: date
                     term_length: free_text
                     termination_clauses: entity_list:free_text
                     … +4 more

corporate_record   Corporate Record
                   specialist: corporate_records_specialist  schema: CorporateRecordExtraction  fields: 7
                     entity_name: name
                     record_type: name
                     effective_date: date
                     key_provisions: entity_list:free_text
                     … +3 more

due_diligence      Due Diligence
                   specialist: due_diligence_specialist  schema: DueDiligenceExtraction  fields: 7
                     target_entity: name
                     diligence_type: name
                     material_findings: entity_list:free_text
          

## The 15-agent roster

In [6]:
roster = lab.agent_roster()
print(f"{len(roster)} agents:")
for name, info in roster.items():
    model = info.get("model") or "(inherits default)"
    role = info.get("role") or "(lane/support agent)"
    print(f"  {name:32s} {model:24s} {role}")

15 agents:
  sorter                           qwen/qwen3.7-flash       sorter (+ sorter_reviewer Lane A, judge-classification)
  sorter_reviewer                  qwen/qwen3.7-flash       sorter (+ sorter_reviewer Lane A, judge-classification)
  arbiter                          qwen/qwen3.7-flash       class specialist (+ judge/arbiter Lane B)
  contracts_specialist             qwen/qwen3.7-flash       (lane/support agent)
  corporate_records_specialist     qwen/qwen3.7-flash       (lane/support agent)
  due_diligence_specialist         qwen/qwen3.7-flash       (lane/support agent)
  correspondence_specialist        qwen/qwen3.7-flash       (lane/support agent)
  compliance_specialist            qwen/qwen3.7-flash       (lane/support agent)
  court_opinions_specialist        qwen/qwen3.7-flash       (lane/support agent)
  insurance_claims_specialist      qwen/qwen3.7-flash       (lane/support agent)
  reporter                         qwen/qwen3.7-flash       reporter
  boss             

## Where to go next

- **01 happy_path_run** — one document through these stations, narrated step by step
- **02 routing_dynamics** — the same doc at five confidence levels: watch the routers choose
- **03 review_lanes** — Lane A and Lane B in action
- **09 all_specialists** — one happy-path run per document class (all 7 specialists)
- **10 edge_cases** — unknown type, missing subtype, $0 amounts, schema-invalid extract, Boss conflict
- **11 huggingface_corpora** — Lucius-Morningstar datasets on the Hub (offline snapshot + live opt-in)